# 010 — Reference CBF designs (EC8-gen2) and analysis files

Designs a ladder of reference concentrically-braced frames and builds a complete OpenSees
analysis folder for each: structural model, modal, monotonic pushover, cyclic pushover
(FEMA 461) and an incremental dynamic analysis (IDA) against the 22-record FEMA P695 far-field
set.

The structural parameters `SITE_CATEGORY`, `N_STOREYS`, `DUCTILITY_CLASS` and `STOREY_HEIGHT`
are each **lists**; the notebook designs every combination of them
(`SITE_CATEGORY × N_STOREYS × DUCTILITY_CLASS × STOREY_HEIGHT`), and each combination sweeps the
full `S_ALPHA_RPS` ladder (`S_alpha,RP` = 0.133 … 1.2). The default set is DC2 / Soil A /
3500 mm storeys at 3 and 5 storeys. (`BAY_WIDTH` was removed — it never reached the designer;
bay geometry lives in `spans_x` in the design template.)

These are *reference* structures: a clean, evenly-spaced set used to characterise collapse
capacity against seismic demand, independent of the site-specific case studies in `011`.

**Upstream:** none — the designs are generated here from `S_ALPHA_RPS` using the `standes`
design algorithm.

**Downstream:** the IDA collapse fragilities of these reference structures feed the
collapse-capacity / hazard comparison work. (The archived SDOF-parameterisation notebook
`012` referenced a *different* design set, `casestudy_designs_ec8_gen2`; this notebook writes
`casestudy_designs_dc2_scA_reference`, so do not assume `012` consumed it.)

> **Scope.** This notebook writes designs to `cfg["models"]["casestudy_designs_dc2_scA_reference"]`
> and analysis folders to `cfg["analysis_data"]["reference_structures_dc2_scA"]`
> (`DEST_ROOT/<tag>/mdof/`). It is separate from `011`, which builds the 60 site-specific
> case studies under a different root. The two do not share output folders.

## Tags

Each structure gets a compact tag `{n}s_dc{dc}_sc{cat}[_h{h}]_{salpha}` (e.g. `3s_dc2_scA_50`).
The `_h{h}` storey-height segment is only added when `STOREY_HEIGHT` has more than one value, so
that when only `N_STOREYS` varies the existing 3-storey tags — and their designs, analysis
folders and any tuned cyclic-pushover `dU` — are reused rather than regenerated.

## Run order

1. Run every cell top to bottom. `SETUP_ANALYSES = True` gates the analysis-file build; set it
   to `False` to (re)design the structures only.
2. The §3 audit reports any cyclic-pushover configs whose `dU` was hand-tuned on disk, so you
   can preserve them (see the note there).
3. Then execute the generated IDA launcher — see the run barrier at the bottom.

Four batch launchers are generated **per combination**, written to
`cfg["scripts"]["batch_run_analyses"]/ec8_gen2_reference_structures/<combo_prefix>/mdof/`
(the folder path encodes the combination, so the filenames are just the analysis type):

| Launcher | Analysis | Template helper |
|---|---|---|
| `po.py` | Monotonic pushover (drift-controlled) | `configure_batch_run_file` |
| `cpo.py` | Cyclic pushover (FEMA 461) | `configure_batch_run_file` |
| `modal.py` | Modal | `configure_batch_run_file` |
| `ida.py` | IDA, FEMA P695 far-field set | `copy_batch_ida_buildings` |

See `phd_project/scripts/templates/Readme.md` for the run-script / config contract.

In [1]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from phd_project.config import config
from phd_project.scripts.case_study_design_scripts.design_site_mdof import (
    design_sites_parallel,
    default_n_workers,
)
from phd_project.scripts.templates.copy_templates_to_folders import (
    copy_nlcbf_model,
    copy_analysis_config,
    configure_batch_run_file,
    copy_batch_ida_buildings,
    copy_file,
)
from phd_project.scripts.case_study_design_scripts.design_file_helpers import (
    get_control_node_from_design_file,
    get_n_primary_modes_from_design_file,
    get_n_damping_modes_from_design_file,
)
from phd_project.scripts.loading_protocols import FEMA_461_loading_protocol
from phd_project.scripts.cpo_du import resolve_cpo_du, audit_cpo_du, existing_cpo_du

cfg = config.load_config()

## 0. Setup & parameters

In [ ]:
# ----------------------------------------------------------------------------
# PARAMETERS
# ----------------------------------------------------------------------------
# Destination root for the (large) analysis folders -- set this to your external
# drive. Folders are written as DEST_ROOT/<tag>/mdof/.
DEST_ROOT = cfg["analysis_data"]["reference_structures_dc2_scA"]

# --- structure / design ------------------------------------------------------
# Each of these is a LIST of values; the notebook designs every combination
# (itertools.product) of them, and each combination sweeps the full S_ALPHA_RPS
# ladder below. Give a single-element list to hold a parameter fixed.
SITE_CATEGORY = ["A"]        # soil class(es) for the MDOF design
N_STOREYS = [3, 5]           # number of storeys
DUCTILITY_CLASS = [2]        # EC8 ductility class
STOREY_HEIGHT = [3500]       # mm
MAX_DESIGN_ITERS = 15

S_ALPHA_RPS = [0.133, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1.0, 1.2]

# --- combination / tag helpers ----------------------------------------------
# Tags stay compact and reuse-friendly: "{n}s_dc{dc}_sc{cat}[_h{h}]_{salpha}".
# The "_h{h}" segment is only added when STOREY_HEIGHT has more than one value,
# so that when only N_STOREYS varies the existing 3s tags (and their designs /
# analysis folders / tuned dU) are byte-identical and get reused.
VARY_HEIGHT = len(set(STOREY_HEIGHT)) > 1


def combo_prefix(n, dc, cat, h):
    """Tag/folder prefix for a parameter combination (no S_alpha suffix)."""
    p = f"{n}s_dc{dc}_sc{cat}"
    if VARY_HEIGHT:
        p += f"_h{h}"
    return p


def make_tag(n, dc, cat, h, salpha):
    """Full building tag: combo prefix + S_alpha suffix."""
    return f"{combo_prefix(n, dc, cat, h)}_{int(salpha * 100)}"


# --- parallelism ---
N_WORKERS = default_n_workers()    # max(cpu_count - 3, 1)

SETUP_ANALYSES = True

if SETUP_ANALYSES:
    # --- monotonic pushover (drift-controlled) ---
    PO_MAX_DRIFT = 6         # %, roof drift limit for the monotonic pushover
    PO_DRIFT_STEP = 0.005    # %, pushover step size

    # --- cyclic pushover (FEMA 461) ---
    CPO_U_MAX = 250          # mm, peak cyclic amplitude (+/-)
    CPO_N_STEPS = 12         # FEMA 461 amplitude steps (12 -> matches example sequence)
    CPO_DU = 0.2             # mm, base displacement ramp step (dU_max)
    CPO_DISPLACEMENTS = np.round(
        np.append(FEMA_461_loading_protocol(CPO_U_MAX, CPO_N_STEPS), 0), 3
    ).tolist()

    # Per-folder dU overrides, {key: dU} where key = "<tag>/mdof" (the folder path
    # relative to DEST_ROOT). The audit cell below prints divergent values ready to
    # paste here. See phd_project/scripts/cpo_du.py.
    CPO_DU_OVERRIDES = {}
    # Keep a finer dU already present on disk instead of overwriting it with CPO_DU.
    # Leave True unless you deliberately want to re-tune from scratch.
    PRESERVE_TUNED_DU = True

    # --- analysis ---
    # NOTE: the MSA parameters (GM_SETS, MAX_N_RECORDS, STRIPE_ORDER_ASCENDING) live in
    # 050_setup_msa_runs_for_complete_sites, which owns the MSA wiring.
    DAMPING_RATIO = 0.05
    DISP_LIMIT = 700           # mm; SDOF collapse displacement limit
    MDOF_DRIFT_LIMIT = 0.2     # MDOF collapse drift limit

    # --- batch launchers ---
    # po / cpo / modal / ida launchers for each combination land under
    # BATCH_BASE/<combo_prefix>/mdof/ (the folder path encodes the combination, so
    # the filenames are just the analysis type). The per-combo BATCH_ROOT is built
    # in the "Build the MDOF analysis folders" cell below.
    BATCH_BASE = (Path(cfg["scripts"]["batch_run_analyses"])
                  / "ec8_gen2_reference_structures")

## 1. Design the reference structures

Designs (or reloads) every parameter combination × `S_ALPHA_RPS` and writes
`site_designs_summary.csv`. Each combination is designed with its own
`design_sites_parallel` call. Already-designed tags are skipped, so this is cheap to re-run and
adding a new combination (e.g. another storey count) only designs the new structures.

In [ ]:
import itertools

design_root = Path(cfg["models"]["casestudy_designs_dc2_scA_reference"])
design_root.mkdir(parents=True, exist_ok=True)
summary_path = design_root / "site_designs_summary.csv"

# One entry per parameter combination; each sweeps the full S_ALPHA_RPS ladder.
combo_specs = []
for cat, n, dc, h in itertools.product(
    SITE_CATEGORY, N_STOREYS, DUCTILITY_CLASS, STOREY_HEIGHT
):
    prefix = combo_prefix(n, dc, cat, h)
    combo_specs.append({
        "site_category": cat,
        "n_storeys": n,
        "ductility_class": dc,
        "storey_height": h,
        "prefix": prefix,
        "tags": [make_tag(n, dc, cat, h, salpha) for salpha in S_ALPHA_RPS],
    })

# tag -> the combination parameters that produced it (used downstream in the dataset)
tag_meta = {
    tag: {k: combo[k] for k in
          ("n_storeys", "ductility_class", "site_category", "storey_height")}
    for combo in combo_specs for tag in combo["tags"]
}

n_tags = sum(len(c["tags"]) for c in combo_specs)
print(f"{len(combo_specs)} combination(s) x {len(S_ALPHA_RPS)} S_alpha = {n_tags} structures")

# only (re)design structures whose tag is not already in the summary csv
existing_df = pd.read_csv(summary_path, index_col="tag") if summary_path.exists() else None
done_tags = set(existing_df.index) if existing_df is not None else set()

# design each combination separately: design_sites_parallel forwards one shared
# kwarg set to every job, so each combination (distinct params) needs its own call.
new_results = []
for combo in combo_specs:
    jobs = [(salpha, tag, design_root)
            for salpha, tag in zip(S_ALPHA_RPS, combo["tags"])]
    pending_jobs = [j for j in jobs if j[1] not in done_tags]
    if not pending_jobs:
        print(f"[{combo['prefix']}] all {len(jobs)} already in {summary_path.name}; skipping")
        continue
    print(f"[{combo['prefix']}] designing {len(pending_jobs)} of {len(jobs)} structures "
          f"({len(jobs) - len(pending_jobs)} already done)")
    new_results.extend(design_sites_parallel(
        pending_jobs,
        n_workers=N_WORKERS,
        site_category=combo["site_category"],
        ductility_class=combo["ductility_class"],
        storey_height=combo["storey_height"],
        n_storeys=combo["n_storeys"],
        max_iters=MAX_DESIGN_ITERS,
    ))

if new_results:
    new_df = pd.DataFrame(new_results).set_index("tag")
    if existing_df is not None:
        existing_df = existing_df.drop(index=new_df.index, errors="ignore")
        designs_df = pd.concat([existing_df, new_df])
    else:
        designs_df = new_df
    designs_df = designs_df.sort_index()
    designs_df.to_csv(summary_path)
else:
    print(f"all {n_tags} structures already in {summary_path.name}; skipping design")
    designs_df = existing_df.sort_index()

designs = designs_df.to_dict("index")

failed = designs_df[~designs_df["success"].fillna(False)]
if len(failed):
    print(f"WARNING: {len(failed)} designs did not succeed:\n{failed.index.tolist()}")

designs_df[["S_alpha_RP", "Vb_coeff", "Wt", "T", "q_design", "success"]]

## 2. Design output dataset

Flatten the design file of each reference structure into a single tidy dataframe -- one row
per building, with the design base shear, seismic mass, behaviour factors, spectrum
parameters and storey forces/shears as columns.

This only reads the design files written in section 1 (no OpenSees analysis needed), keeping
just the structures that designed successfully. The dataframe is pickled to
`cfg["proc_data"]["dc2_scA_reference_dataset"]`.

In [ ]:
# Flatten each design file into one row: mapping of {column: nested-json path}.
design_out_parameter_map = {
    "roof_height": "['structure']['level_coordinates'][-1]",
    "V_wind_GQWSI_LC1": "['nonseismic_design_outputs']['GQWSI_LC1']['uls_wind_base_shear']",
    "V_wind_GQWSI_LC2": "['nonseismic_design_outputs']['GQWSI_LC2']['uls_wind_base_shear']",
    "V_wind_GQWSI_LC3": "['nonseismic_design_outputs']['GQWSI_LC3']['uls_wind_base_shear']",
    "T1_GQWSI_LC1": "['nonseismic_design_outputs']['GQWSI_LC1']['period']",
    "T1_GQWSI_LC2": "['nonseismic_design_outputs']['GQWSI_LC2']['period']",
    "T1_GQWSI_LC3": "['nonseismic_design_outputs']['GQWSI_LC3']['period']",
    "gravity_frame_elastic_baseshear": "['seismic_design_outputs']['gravity_frame_elastic_baseshear']",
    "gravity_design_equivalent_seismic_baseshear": "['seismic_design_outputs']['gravity_design_equivalent_seismic_baseshear']",
    "seismic_mass": "['seismic_design_outputs']['seismic_mass']",
    "ductility_class": "['seismic_design_outputs']['ductility_class']",
    "vertical_regularity": "['seismic_design_outputs']['vertical_regularity']",
    "q_design": "['seismic_design_outputs']['q_design']",
    "q_D": "['seismic_design_outputs']['q_D']",
    "q_R": "['seismic_design_outputs']['q_R']",
    "q_S": "['seismic_design_outputs']['q_S']",
    "q_max": "['seismic_design_outputs']['q_max']",
    "design_period": "['seismic_design_outputs']['design_period']",
    "design_spectral_acceleration": "['seismic_design_outputs']['design_spectral_acceleration']",
    "design_baseshear": "['seismic_design_outputs']['design_baseshear']",
    "lambda": "['seismic_design_outputs']['lambda']",
    "S_alpha_RP": "['seismic_design_outputs']['spectrum_parameters']['S_alpha_RP']",
    "S_beta_RP": "['seismic_design_outputs']['spectrum_parameters']['S_beta_RP']",
    "S_alpha": "['seismic_design_outputs']['spectrum_parameters']['S_alpha']",
    "S_beta": "['seismic_design_outputs']['spectrum_parameters']['S_beta']",
    "T_beta": "['seismic_design_outputs']['spectrum_parameters']['T_beta']",
    "T_A": "['seismic_design_outputs']['spectrum_parameters']['T_A']",
    "T_B": "['seismic_design_outputs']['spectrum_parameters']['T_B']",
    "T_C": "['seismic_design_outputs']['spectrum_parameters']['T_C']",
    "T_D": "['seismic_design_outputs']['spectrum_parameters']['T_D']",
    "F_alpha": "['seismic_design_outputs']['spectrum_parameters']['F_alpha']",
    "F_beta": "['seismic_design_outputs']['spectrum_parameters']['F_beta']",
    "F_T": "['seismic_design_outputs']['spectrum_parameters']['F_T']",
    "F_A": "['seismic_design_outputs']['spectrum_parameters']['F_A']",
    "site_category": "['seismic_design_outputs']['spectrum_parameters']['site_category']",
    "S_delta": "['seismic_design_outputs']['spectrum_parameters']['S_delta']",
    "delta": "['seismic_design_outputs']['spectrum_parameters']['delta']",
    "seismic_action_class": "['seismic_design_outputs']['spectrum_parameters']['seismic_action_class']",
}


def read_out_value(data, path):
    """Pull a value out of the nested design dict, returning NaN if the path is absent.

    `path` is a string of chained subscripts, e.g. "['structure']['level_coordinates'][-1]".
    """
    try:
        return eval("data" + path)
    except Exception:
        return np.nan


MAX_STOREYS = max(N_STOREYS)  # width of the seismic_F* / seismic_V* columns

building_data_dicts = []
for tag in [t for t in designs if designs[t]["success"]]:
    with open(designs[tag]["design_out_json"], "r") as f:
        data = json.load(f)

    building_data = {"name": tag}
    for df_tag, dd_path in design_out_parameter_map.items():
        building_data[df_tag] = read_out_value(data, dd_path)

    # Guarantee the sweep parameters as columns straight from the combination that
    # produced this tag (independent of the design-file contents). This also
    # overrides site_category / ductility_class read via the map above with the
    # authoritative combination values, and adds n_storeys / storey_height. Falls
    # back to parsing the storey count from the tag for any row predating tag_meta.
    meta = tag_meta.get(tag)
    if meta is not None:
        building_data.update(meta)
    else:
        building_data["n_storeys"] = int(tag.split("s", 1)[0])

    # design base-shear coefficient Vb/Wt (Wt = seismic_mass * g), carried straight
    # from the design summary so downstream notebooks read it without recomputing.
    # See phd_project/scripts/case_study_design_scripts/design_site_mdof.py.
    building_data["Vb"] = designs[tag].get("Vb")
    building_data["Wt"] = designs[tag].get("Wt")
    building_data["Vb_coeff"] = designs[tag].get("Vb_coeff")

    # design storey forces and the cumulative storey shears
    storey_forces = data["seismic_design_outputs"]["storey_forces"]
    storey_shears = np.cumsum(storey_forces)
    for ii in range(MAX_STOREYS):
        building_data[f"seismic_F{ii + 1}"] = (
            storey_forces[ii] if ii < len(storey_forces) else np.nan
        )
        building_data[f"seismic_V{ii + 1}"] = (
            storey_shears[ii] if ii < len(storey_shears) else np.nan
        )

    building_data_dicts.append(building_data)

building_data_df = pd.DataFrame(building_data_dicts)

building_data_df = building_data_df.sort_values(
    by=["n_storeys", "S_alpha_RP"]
).reset_index(drop=True)

dataset_path = cfg["proc_data"]["dc2_scA_reference_dataset"]
with open(dataset_path, "wb") as f:
    pickle.dump(building_data_df, f)
print(f"wrote {dataset_path}  ({len(building_data_df)} rows)")

building_data_df.head()

## 3. Cyclic-pushover `dU` — audit before overwriting

Some models need a finer displacement increment than the `CPO_DU` default to converge.
Rebuilding a folder rewrites its `config_cyclic_pushover.py`, which would throw that tuning
away. The cell below reports every existing config whose `dU` differs from what would be
written. With `PRESERVE_TUNED_DU = True` (the default) those values are kept; to pin them
explicitly, paste the printed snippet into `CPO_DU_OVERRIDES`.

In [ ]:
if SETUP_ANALYSES:
    all_tags = [tag for combo in combo_specs for tag in combo["tags"]]
    cpo_items = [
        (f"{tag}/mdof", DEST_ROOT / tag / "mdof" / "config_cyclic_pushover.py")
        for tag in all_tags if designs[tag]["success"]
    ]
    divergent = audit_cpo_du(cpo_items, CPO_DU, CPO_DU_OVERRIDES)

    if divergent:
        print(f"{len(divergent)} existing config(s) use a dU different from the default "
              f"({CPO_DU}):")
        for key, du in sorted(divergent.items()):
            print(f"   {key:28s} dU = {du}")
        print("\nPRESERVE_TUNED_DU =", PRESERVE_TUNED_DU,
              "-> these will be KEPT." if PRESERVE_TUNED_DU else "-> these will be OVERWRITTEN.")
        print("\nTo pin these explicitly, paste into CPO_DU_OVERRIDES:")
        print("CPO_DU_OVERRIDES = {")
        for key, du in sorted(divergent.items()):
            print(f'    "{key}": {du},')
        print("}")
    else:
        print("no divergent dU values found")

## 4. Build the MDOF analysis folders

One `DEST_ROOT/<tag>/mdof/` per successful design, then the top-level IDA launcher. The
cyclic-pushover `dU` for each folder is resolved by `resolve_cpo_du` (override → tuned value
on disk → `CPO_DU`).

In [ ]:
if SETUP_ANALYSES:
    # Name of the structural model file written into each analysis folder. The configs
    # import the model by this name (their `model_file_name` variable), so it no longer
    # has to be "structural_model.py".
    REDUCED_RECORDERS = True

    def add_cyclic_pushover_files(folder: Path, ctrl_node: int, key: str,
                                  model_file_name: str) -> Path:
        """Add a FEMA 461 cyclic pushover run script + config to an analysis folder.
        The config imports the structural model from `model_file_name` in the folder itself.
        `key` (e.g. "3s_dc2_scA_30/mdof") selects any per-folder dU override and lets a
        tuned dU already on disk be preserved -- see phd_project/scripts/cpo_du.py."""
        copy_file(cfg["templates"]["run_cyclic_pushover"], folder / "run_cyclic_pushover.py")
        cfg_dst = folder / "config_cyclic_pushover.py"
        du = resolve_cpo_du(key, cfg_dst, CPO_DU_OVERRIDES, CPO_DU, PRESERVE_TUNED_DU)
        copy_analysis_config(
            cfg["templates"]["config_cyclic_pushover"],
            cfg_dst,
            results_folder_name="cyclic_pushover",
            model_file_name=model_file_name,
            update_config={
                "displacement_type": "displacement",
                "dU": du,
                "ctrl_node": ctrl_node,
                "displacements": CPO_DISPLACEMENTS,
            },
        )
        return cfg_dst

    def build_mdof_folder(tag: str) -> Path:
        folder = DEST_ROOT / f"{tag}" / "mdof"
        folder.mkdir(parents=True, exist_ok=True)

        # design file (structural_model reads it from its own folder)
        design_out = Path(designs[tag]["design_out_json"])
        design_dst = folder / f"{tag}_designfile.json"
        copy_file(design_out, design_dst)

        # structural model # full recorders
        n_damping_modes = get_n_damping_modes_from_design_file(design_dst)
        init_fn_full = copy_nlcbf_model(
            cfg["templates"],
            folder,
            design_json=design_dst.name,
            damping_updates={"n_modes": n_damping_modes, "damping_ratio": DAMPING_RATIO},
            recorder_updates={"drift_limit": MDOF_DRIFT_LIMIT},
        )
        
        # structural model # reduced recorders
        n_damping_modes = get_n_damping_modes_from_design_file(design_dst)
        init_fn_reduced = copy_nlcbf_model(
            cfg["templates"],
            folder,
            design_json=design_dst.name,
            damping_updates={"n_modes": n_damping_modes, "damping_ratio": DAMPING_RATIO},
            recorder_updates={"drift_limit": MDOF_DRIFT_LIMIT},
            reduced=REDUCED_RECORDERS
        )

        # modal analysis (for participation factor)
        copy_file(cfg["templates"]["run_modal"], folder / "run_modal.py")
        copy_analysis_config(
            cfg["templates"]["config_modal"],
            folder / "config_modal.py",
            results_folder_name="modal",
            model_file_name=init_fn_full,
            update_config={"n_modes": get_n_primary_modes_from_design_file(design_dst)},
        )

        ctrl_node = get_control_node_from_design_file(design_dst)

        # monotonic pushover (drift-controlled)
        copy_file(cfg["templates"]["run_pushover"], folder / "run_pushover.py")
        copy_analysis_config(
            cfg["templates"]["config_pushover"],
            folder / "config_pushover.py",
            results_folder_name="pushover",
            model_file_name=init_fn_full,
            update_config={
                "displacement_type": "drift",
                "U_max": PO_MAX_DRIFT,
                "dU": PO_DRIFT_STEP,
                "ctrl_node": ctrl_node,
            },
        )

        # cyclic pushover (FEMA 461)
        add_cyclic_pushover_files(
            folder,
            ctrl_node,
            key=folder.relative_to(DEST_ROOT).as_posix(),
            model_file_name=init_fn_full,
        )
        
        # ida analysis
        # copy files that don't need changing:
        src_proc = cfg["templates"]["ida_process_recorder_roof_drift"]
        dst_proc = folder / "ida_process_recorders.py"
        copy_file(src_proc, dst_proc)

        src_im = cfg["templates"]["config_im_SA"]
        dst_im = folder / "config_im_SA.py"
        copy_file(src_im, dst_im)

        src_inj = cfg["templates"]["nltha_injection_update_damping"]
        dst_inj = folder / "injection_functions.py"
        copy_file(src_inj, dst_inj)

        src = cfg["templates"]["run_batch_ida_per_record"]
        dst = folder / "run_batch_ida_per_record.py"
        copy_file(src, dst)

        src = cfg["templates"]["run_ida_htf_per_record"]
        dst = folder / "run_ida_htf_per_record.py"
        copy_file(src, dst)

        # now copy the config 
        src_config = cfg["templates"]["config_ida_htf"]
        dst_ida_config = folder / "config_ida_htf_femap695.py"

        changes = {
            "results_folder_name": "ida_femap695",
            "model_file_name": init_fn_reduced,
            "do_fill": True,
            "gm_json_src_str" : 'D:/gm_records_p695',
            "record_filenames": ['fema_p695_120111.json',
                                'fema_p695_120121.json',
                                'fema_p695_120411.json',
                                'fema_p695_120521.json',
                                'fema_p695_120611.json',
                                'fema_p695_120621.json',
                                'fema_p695_120711.json',
                                'fema_p695_120721.json',
                                'fema_p695_120811.json',
                                'fema_p695_120821.json',
                                'fema_p695_120911.json',
                                'fema_p695_120921.json',
                                'fema_p695_121011.json',
                                'fema_p695_121021.json',
                                'fema_p695_121111.json',
                                'fema_p695_121211.json',
                                'fema_p695_121221.json',
                                'fema_p695_121321.json',
                                'fema_p695_121411.json',
                                'fema_p695_121421.json',
                                'fema_p695_121511.json',
                                'fema_p695_121711.json',
                                ],
        }

        copy_analysis_config(src_config, dst_ida_config, **changes)

        return folder, dst_ida_config

    def _jobs(mdof_folders, script_name, config_name, suffix):
        return [{"script": folder / script_name,
                 "config": [folder / config_name],
                 "name": [f"{tag}_{suffix}"]}
                for tag, (folder, _ida_config) in mdof_folders.items()]

    # Build folders + batch launchers per combination. Each combination's launchers
    # land under BATCH_BASE/<combo_prefix>/mdof/, so the folder path encodes the
    # combination and the filenames stay just the analysis type.
    for combo in combo_specs:
        combo_tags = [tag for tag in combo["tags"] if designs[tag]["success"]]
        if not combo_tags:
            print(f"[{combo['prefix']}] no successful designs; skipping")
            continue

        mdof_folders = {tag: build_mdof_folder(tag) for tag in tqdm(
            combo_tags, desc=f"Building MDOF folders [{combo['prefix']}]")}
        print(f"[{combo['prefix']}] built {len(mdof_folders)} MDOF folders")

        BATCH_ROOT = BATCH_BASE / combo["prefix"] / "mdof"
        BATCH_ROOT.mkdir(parents=True, exist_ok=True)

        # po / cpo / modal use the heterogeneous-jobs launcher template
        for filename, jobs in [
            ("po.py", _jobs(mdof_folders, "run_pushover.py", "config_pushover.py", "po")),
            ("cpo.py", _jobs(mdof_folders, "run_cyclic_pushover.py", "config_cyclic_pushover.py", "cpo")),
            ("modal.py", _jobs(mdof_folders, "run_modal.py", "config_modal.py", "modal")),
        ]:
            configure_batch_run_file(cfg["templates"]["batch_run"], BATCH_ROOT / filename, jobs)
            print(f"[{combo['prefix']}] wrote {filename} ({len(jobs)} jobs)")

        # ida uses the multi-building coordinator launcher (a different template)
        ida_buildings = [{"folder": folder, "config": ida_config}
                         for folder, ida_config in mdof_folders.values()]
        copy_batch_ida_buildings(
            cfg["templates"]["run_batch_ida_buildings"],
            BATCH_ROOT / "ida.py",
            ida_buildings,
        )
        print(f"[{combo['prefix']}] wrote ida.py ({len(ida_buildings)} buildings)")
        print(f"[{combo['prefix']}] launchers -> {BATCH_ROOT}\n")
    

---

# ⛔ RUN BARRIER — run the reference analyses

The analysis folders and the four batch launchers **per combination** are now written, under
`cfg["scripts"]["batch_run_analyses"]/ec8_gen2_reference_structures/<combo_prefix>/mdof/`
(e.g. `.../3s_dc2_scA/mdof/` and `.../5s_dc2_scA/mdof/`). Run each combination's launchers from
its own folder:

```
python modal.py     # fast
python po.py        # monotonic pushover
python cpo.py       # cyclic pushover (FEMA 461)
python ida.py       # IDA, FEMA P695 set -- slow (multi-building coordinator)
```

Each `configure_batch_run_file` launcher (`po`/`cpo`/`modal`) opens one console per structure;
`ida.py` is the multi-building coordinator. Results land under each
`DEST_ROOT/<tag>/mdof/` in `pushover/`, `cyclic_pushover/`, `modal/` and `ida_femap695/`.
Concurrency is capped globally by `phd_project/process_semaphore/process_semaphore.py`, so the
launchers can be started back to back.

> **Note.** The combinatorial refactor moved the 3-storey launchers from the old
> `.../3s/mdof/` path to `.../3s_dc2_scA/mdof/`. The stale launchers under the old `3s/` folder
> are harmless leftovers and can be deleted manually.